# OCC SOXX/SOXL 每日看涨看跌来源完整性

假设只涉及来源资格：在 2024-11-01 至 2026-10-21 的 Qlib 交易日上，OCC 各清算账户侧数量应对 Cboe 相同交易所合约量精确呈 2:1。源计划、请求清单、预算和哈希先冻结。无价格收益、训练、账户或期权方向性结论。

In [ ]:
from pathlib import Path
import os, csv, json, hashlib
from collections import defaultdict
import pandas as pd
os.environ.setdefault("MLFLOW_ALLOW_FILE_STORE", "true")
root = Path.cwd().resolve()
while not (root / "SPEC.md").is_file():
    if root.parent == root:
        raise RuntimeError("workspace root missing")
    root = root.parent
card = root / "research/occ_put_call_daily_source_20261022"
plan = json.loads((card / "plan.json").read_text())
manifest = json.loads((card / "request_manifest.json").read_text())
qual = json.loads((card / "qualification_plan.json").read_text())
extension = json.loads((card / "qualification_source_extension_plan.json").read_text())
sha = lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan["frozen"] and plan["execution_authorized"]
assert manifest["plan_sha256"] == qual["source_plan_sha256"] == sha(card / "plan.json")
assert qual["request_manifest_sha256"] == sha(card / "request_manifest.json")
assert extension["qualification_plan_sha256"] == sha(card / "qualification_plan.json")
assert not (card / "qualification_run_state.json").exists(), "one frozen qualification attempt already ran"
fetch_state = json.loads((card / "fetch_run_state.json").read_text())
assert fetch_state["status"] == "completed" and fetch_state["completed_http_gets"] == manifest["planned_http_gets"]
receipts = [json.loads(line) for line in (card / "fetch_receipts.jsonl").read_text().splitlines() if line]
requests = [x for x in manifest["items"] if "reuse_sha256" not in x]
assert len(receipts) == len(requests) == manifest["planned_http_gets"]
receipt_map = {}
for index, (request, receipt) in enumerate(zip(requests,receipts)):
    assert receipt["status"] == "ok" and receipt["index"] == index
    assert (receipt["symbol"],receipt["date"],receipt["url"]) == (request["symbol"],request["date"],request["url"])
    receipt_map[(receipt["symbol"],receipt["date"])] = receipt
cboe = {}
for symbol in plan["symbols"]:
    source = root / "data/snapshots/cboe_total_options_activity_source_20261022" / (symbol.lower()+"_2019-01-02_2026-10-21.bin")
    assert sha(source) == qual["cboe_raw_sha256"][symbol]
    with source.open(newline="") as stream:
        for row in csv.DictReader(stream):
            date = pd.Timestamp(row["Trade Date"]).date().isoformat()
            if date >= plan["first_trade_date"] and date <= plan["last_trade_date"]:
                key = (symbol,date,row["Exchange"])
                assert key not in cboe
                cboe[key] = int(row["Volume"])
    q = extension["forward_cboe_sources"][symbol]
    following = root / q["path"]
    assert sha(following) == q["sha256"]
    with following.open(newline="") as stream:
        for row in csv.DictReader(stream):
            if row["Trade Date"] == "2026/10/21":
                key = (symbol,"2026-10-21",row["Exchange"])
                assert key not in cboe
                cboe[key] = int(row["Volume"])
from src.data.occ_put_call_daily_source import parse_daily
shared = ["CBOE","BATS","C2","EDGX"]
daily, mismatches, missing_types, odd_types, zero_totals = [], [], [], [], []
for item in manifest["items"]:
    symbol,date = item["symbol"],item["date"]
    if "reuse_sha256" in item:
        path = root / item["reuse_relative_path"]
        assert sha(path) == item["reuse_sha256"]
    else:
        receipt = receipt_map[(symbol,date)]
        path = root / receipt["relative_path"]
        assert sha(path) == receipt["sha256"]
    body = path.read_bytes()
    parsed = parse_daily(body,symbol,date)
    by_exchange = defaultdict(int)
    with path.open(newline="") as stream:
        next(csv.reader(stream))
        for row in csv.reader(stream):
            by_exchange[row[5]] += int(row[0])
    for venue in shared:
        observed,reference = by_exchange[venue],cboe.get((symbol,date,venue),0)
        if observed != 2*reference:
            mismatches.append({"symbol":symbol,"date":date,"exchange":venue,
                               "occ_quantity":observed,"cboe_contracts":reference})
    if parsed["option_types_present"] != ["C","P"]:
        missing_types.append({"symbol":symbol,"date":date,"types":parsed["option_types_present"]})
    if parsed["call_quantity"]%2 or parsed["put_quantity"]%2:
        odd_types.append({"symbol":symbol,"date":date,"call":parsed["call_quantity"],"put":parsed["put_quantity"]})
    total = parsed["call_quantity"]+parsed["put_quantity"]
    if total == 0:
        zero_totals.append({"symbol":symbol,"date":date})
    daily.append({"date":date,"symbol":symbol,"call_contracts":parsed["call_quantity"]/2,
                  "put_contracts":parsed["put_quantity"]/2,"total_contracts":total/2,
                  "put_share":parsed["put_quantity"]/total if total else float("nan"),"roots":parsed["roots"],
                  "venue_count":len(parsed["venues"]),"source_sha256":sha(path)})
assert len(daily) == manifest["planned_pairs"]
rows = pd.DataFrame(daily).sort_values(["date","symbol"]).reset_index(drop=True)
assert len(rows.groupby("date")) == manifest["calendar_sessions"]
status = "source_coverage_qualified_with_PIT_gap" if not mismatches else "source_reconciliation_failed"
result = {"status":status,"calendar_sessions":manifest["calendar_sessions"],
          "symbol_date_pairs":len(rows),"http_gets":manifest["planned_http_gets"],
          "reused_pairs":manifest["reused_pairs"],"shared_venue_day_comparisons":len(rows)*len(shared),
          "mismatch_count":len(mismatches),"odd_call_put_quantity_count":len(odd_types),
          "one_option_type_only_count":len(missing_types),"zero_total_days":len(zero_totals),
          "first_date":str(rows.date.min()),"last_date":str(rows.date.max()),
          "historical_first_public_time_qualified":False,"two_complete_economic_years_proven":False,
          "model_fits":0,"accounts":0,"market_outcome_reads":0,"new_D_calls":0,
          "boundary":"OCC account-side quantity is exactly 2x Cboe contracts only if every shared venue-day matches. The put share cancels common side duplication. Raw files are retrospective, no original publication vintage or return label has been verified."}
(card / "qualification_run_state.json").write_text(json.dumps({"status":"started","qualification_plan_sha256":sha(card / "qualification_plan.json")})+chr(10))
import qlib
from qlib.workflow import R
qlib.init(provider_uri=str(root / plan["calendar_provider"]),region="us",kernels=1)
tracking = card / "qlib_recorder"
with R.start(experiment_name=plan["experiment"],uri=str(tracking)):
    rec=R.get_recorder()
    rec.save_objects(artifact_path="sources", plan=plan, request_manifest=manifest,
                     acquisition_receipts=receipts, qualification_plan=qual,
                     Cboe_extension=extension)
    rec.save_objects(artifact_path="analysis", qualification=result,daily=rows,
                     mismatches=pd.DataFrame(mismatches),missing_option_type_days=pd.DataFrame(missing_types),
                     odd_quantity_days=pd.DataFrame(odd_types),zero_total_days=pd.DataFrame(zero_totals))
    rec.log_metrics(source_sessions=result["calendar_sessions"], source_pairs=result["symbol_date_pairs"],
                    matched_venue_day_comparisons=result["shared_venue_day_comparisons"],
                    source_mismatches=result["mismatch_count"], model_fits=0, accounts=0)
    result.update(recorder_id=rec.id,experiment_id=str(rec.experiment_id),
                  recorder_uri=str(tracking),recorder_local_dir=str(tracking/str(rec.experiment_id)/rec.id))
artifacts=Path(result["recorder_local_dir"])/"artifacts"
verification={"recorder_id":result["recorder_id"],"entries":[{"path":str(path.relative_to(card)),
    "sha256":sha(path),"bytes":path.stat().st_size} for path in sorted(artifacts.rglob("*")) if path.is_file()]}
(card / "verification_manifest.json").write_text(json.dumps(verification,indent=2)+chr(10))
(card / "source_qualification_result.json").write_text(json.dumps(result,indent=2)+chr(10))
summary={"experiment":plan["experiment"],"status":status,"decision":"source_qualified_information_screen_needs_PIT_and_design" if status.startswith("source_coverage_qualified") else "source_not_qualified_reconciliation_failed",
         "result":"source_qualification_result.json","recorder_id":result["recorder_id"],
         "experiment_id":result["experiment_id"],"recorder_uri":result["recorder_uri"],
         "recorder_local_dir":result["recorder_local_dir"],"model_fits":0,"accounts":0,
         "qualified_forward_days":0,"objective_achieved":False,"artifact_policy":plan["artifact_policy"]}
(card / "summary.json").write_text(json.dumps(summary,indent=2)+chr(10))
(card / "conclusion.md").write_text("# 中文结论\n\nOCC 看涨/看跌每日来源覆盖 "+str(result["calendar_sessions"])+" 个 Qlib 交易日、两标的共 "+str(result["symbol_date_pairs"])+" 个源对象。与 Cboe 四个共同交易所逐日核对 "+str(result["shared_venue_day_comparisons"])+" 项，数量不符 "+str(result["mismatch_count"])+" 项，奇数买卖权清算数量 "+str(result["odd_call_put_quantity_count"])+" 项。Recorder `"+result["recorder_id"]+"` 保存完整逐日表与差异。\n\n原始数据是后验下载，不证明当年首次公开时点或修订版本；没有读取收益标签、拟合模型或运行账户。本来源即使逐日完整，也只是信息检验的前提，不能宣称稳定跑赢 SOXX。下一步先核实发布时序，再按独立冻结的设计做相对既有价格/波动基线的信息筛选。\n")
(card / "qualification_run_state.json").write_text(json.dumps({"status":"completed","recorder_id":result["recorder_id"],"qualification_plan_sha256":sha(card / "qualification_plan.json")})+chr(10))
print(json.dumps({"status":status,"sessions":result["calendar_sessions"],"pairs":len(rows),
                  "mismatches":len(mismatches),"odd_types":len(odd_types),"recorder_id":result["recorder_id"]}))

## 中文结论

上方代码保存原生 Qlib Recorder、来源完整性和逐日对账。`conclusion.md` 报告资格与限制；发布时序、信息增益及 SOXX 经济目标是后续独立门槛。